# 05. Новости: выгрузка GDELT по регионам России (2022–2024)

GDELT публикует ежедневный файл со всеми событиями, извлечёнными из мировых новостей. У каждого события
есть место (страна и регион первого уровня), тип по классификации CAMEO, тональность и число статей.

Скрипт скачивает дневные файлы, оставляет только события в России и сводит их в таблицу
**«регион GDELT × месяц»**. Привязку регионов GDELT к нашим регионам и МО делает основной пайплайн.

Дневные выгрузки кешируются в `data/raw/gdelt_cache/`; повторный расчёт использует готовые дни.
Результаты: `data/raw/gdelt_ru_region_month.parquet` и `data/external/gdelt_ru_regions.csv`.

In [ ]:
import io, time, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
import requests

START, END = "2022-01-01", "2024-12-31"
URL = "http://data.gdeltproject.org/events/{d}.export.CSV.zip"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"; RAW.mkdir(parents=True, exist_ok=True)
EXTERNAL = ROOT / "data" / "external"; EXTERNAL.mkdir(parents=True, exist_ok=True)
CACHE = RAW / "gdelt_cache"; CACHE.mkdir(exist_ok=True)
THREADS = 8

COLS = {1: "SQLDATE", 26: "EventCode", 28: "EventRootCode", 29: "QuadClass", 30: "GoldsteinScale",
        31: "NumMentions", 32: "NumSources", 33: "NumArticles", 34: "AvgTone",
        49: "ActionGeo_Type", 50: "ActionGeo_FullName", 51: "ActionGeo_CountryCode", 52: "ActionGeo_ADM1Code"}

def fetch(day):
    out = CACHE / f"{day}.parquet"
    if out.exists(): return day, "cache"
    for attempt in range(3):
        try:
            r = requests.get(URL.format(d=day), timeout=120)
            if r.status_code == 404: return day, "нет файла"
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                raw = z.read(z.namelist()[0])
            df = pd.read_csv(io.BytesIO(raw), sep="\t", header=None, dtype=str, usecols=list(COLS),
                             on_bad_lines="skip").rename(columns=COLS)
            df = df[df.ActionGeo_CountryCode == "RS"]                # Россия (код FIPS)
            df.to_parquet(out, index=False)
            return day, len(df)
        except Exception as e:
            err = e; time.sleep(5)
    return day, f"ошибка: {err}"

Проверка загрузки и структуры дневной выгрузки за 15 июня 2023 года.

In [ ]:
print(fetch("20230615"))
test = pd.read_parquet(CACHE / "20230615.parquet")
print(test.shape); print(test.head(3).T)

In [ ]:
days = pd.date_range(START, END).strftime("%Y%m%d")
status, t0 = {}, time.time()
with ThreadPoolExecutor(THREADS) as ex:
    futs = [ex.submit(fetch, d) for d in days]
    for i, f in enumerate(as_completed(futs), 1):
        d, s = f.result(); status[d] = s
        if i % 50 == 0: print(f"{i}/{len(days)} дней, {time.time() - t0:.0f} c")
bad = {d: s for d, s in status.items() if isinstance(s, str) and s != "cache"}
print("проблемные дни:", len(bad), list(bad.items())[:5])

Агрегация «регион × месяц». Метрики:
- `n_events`, `n_articles` — объём новостного потока;
- `tone_mean` — средняя тональность (взвешенная по числу статей; меньше нуля — негатив);
- `goldstein_mean` — шкала Голдштейна: от −10 (конфликт) до +10 (сотрудничество);
- `share_conflict` — доля событий «материальный конфликт» (QuadClass = 4);
- `share_protest` — протесты (CAMEO 14); `share_violence` — нападения, бои, массовое насилие (CAMEO 18–20);
- `share_econ` — экономические события: санкции, эмбарго, экономическая помощь (коды 163, 1031, 1041, 0231, 0331, 061).

In [ ]:
ev = pd.concat([pd.read_parquet(p) for p in sorted(CACHE.glob("*.parquet"))], ignore_index=True)
for c in ["QuadClass", "GoldsteinScale", "NumMentions", "NumArticles", "AvgTone"]:
    ev[c] = pd.to_numeric(ev[c], errors="coerce")
ev["date"] = pd.to_datetime(ev.SQLDATE, format="%Y%m%d", errors="coerce").dt.to_period("M").dt.to_timestamp()
ev = ev[ev.date.between(START, END) & ev.ActionGeo_ADM1Code.notna() & (ev.ActionGeo_ADM1Code != "RS")]
root = pd.to_numeric(ev.EventRootCode, errors="coerce")
ev["is_conflict"] = (ev.QuadClass == 4).astype(int)
ev["is_protest"] = (root == 14).astype(int)
ev["is_violence"] = root.between(18, 20).astype(int)
ev["is_econ"] = ev.EventCode.astype(str).isin(["163", "1031", "1041", "0231", "0331", "061"]).astype(int)
ev["tone_w"] = ev.AvgTone * ev.NumArticles

agg = ev.groupby(["ActionGeo_ADM1Code", "date"]).agg(
    n_events=("SQLDATE", "size"), n_articles=("NumArticles", "sum"), tone_w=("tone_w", "sum"),
    goldstein_mean=("GoldsteinScale", "mean"), share_conflict=("is_conflict", "mean"),
    share_protest=("is_protest", "mean"), share_violence=("is_violence", "mean"),
    share_econ=("is_econ", "mean")).reset_index()
agg["tone_mean"] = agg.tone_w / agg.n_articles
agg = agg.drop(columns="tone_w").rename(columns={"ActionGeo_ADM1Code": "gdelt_adm1"})
agg.to_parquet(RAW / "gdelt_ru_region_month.parquet", index=False)

names = (ev[ev.ActionGeo_Type == "5"].groupby("ActionGeo_ADM1Code").ActionGeo_FullName
         .agg(lambda s: s.value_counts().index[0]).rename("full_name").reset_index()
         .rename(columns={"ActionGeo_ADM1Code": "gdelt_adm1"}))
names = names.merge(agg.groupby("gdelt_adm1").n_events.sum().reset_index(), on="gdelt_adm1", how="right")
names.to_csv(EXTERNAL / "gdelt_ru_regions.csv", index=False)
print(agg.shape, "| регионов:", agg.gdelt_adm1.nunique(), "| месяцев:", agg.date.nunique())
print(names.sort_values("n_events", ascending=False).head(10).to_string(index=False))